# Cost Model Comparison (grid:astar)

Compare all cost models using `grid:astar` across weights 1, 10, 100.
This notebook generates 20 elevation samples and visualizes resulting paths.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from passage.generate import find_path, load_tile_grid
from passage.pathfinding_utils import COST_MODELS
from passage.visualization import tensor_to_image

In [ ]:
# Configuration
CONFIG_PATH = ROOT / "config" / "config.yaml"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

RESOLUTION = 256
NUM_SAMPLES = 20
WEIGHTS = [1.0, 10.0, 100.0]
COST_MODELS_LIST = sorted(list(COST_MODELS))

print(f"Resolution: {RESOLUTION}")
print(f"Weights: {WEIGHTS}")
print(f"Cost models: {COST_MODELS_LIST}")

In [ ]:
# Load calibration data
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")
print(f"Number of tiles: {len(tile_stats_df)}")

In [ ]:
import math
import random

TILE_SIZE_PX = 3600


def generate_random_crop(
    resolution: int,
    tile_stats_df: pd.DataFrame,
    tiles_dirpath: Path,
    global_min: float,
    global_max: float,
    seed: int | None = None,
) -> np.ndarray | None:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    half_res = resolution // 2
    half_tiles = max(1, math.ceil(half_res / TILE_SIZE_PX))

    for _ in range(10):
        target_elev = random.uniform(global_min, global_max)
        candidates = tile_stats_df[(tile_stats_df["min"] <= target_elev) & (tile_stats_df["max"] >= target_elev)]
        if candidates.empty:
            continue
        selected = candidates.sample(n=1).iloc[0]
        center_lat = int(selected["lat"])
        center_lon = int(selected["lon"])
        grid = load_tile_grid(tiles_dirpath, center_lat, center_lon, half_tiles=half_tiles)
        grid_h, grid_w = grid.shape
        if grid_h < resolution or grid_w < resolution:
            continue
        y1 = random.randint(0, grid_h - resolution)
        x1 = random.randint(0, grid_w - resolution)
        crop = grid[y1 : y1 + resolution, x1 : x1 + resolution]
        return crop
    return None


def get_marker_positions(resolution: int) -> tuple[tuple[int, int], tuple[int, int]]:
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    return (pos1, pos1), (pos2, pos2)

In [ ]:
def compute_paths_for_crop(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int], cost_models: list[str], weights: list[float]
) -> dict:
    results = {}
    for cm in cost_models:
        results[cm] = {}
        for w in weights:
            solver_config = {
                "name": "grid:astar",
                "diagonal_movement": "always",
                "weight": float(w),
                "cost_model": cm,
                "heuristic": "euclidean",
            }
            path = find_path(crop, start, end, solver_config)
            results[cm][w] = path
    return results


def make_tensor_vis(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int]
) -> tuple[np.ndarray, tuple[float, float]]:
    # Use raw elevation with provided range for consistent normalization in visualization
    elev_min = float(np.min(crop))
    elev_max = float(np.max(crop))
    marker_channel = np.zeros_like(crop, dtype=np.float32)
    marker_channel[start] = 1.0
    marker_channel[end] = 1.0
    obstacles_channel = np.zeros_like(crop, dtype=np.float32)
    tensor_vis = np.stack([crop.astype(np.float32), marker_channel, obstacles_channel], axis=-1)
    return tensor_vis, (elev_min, elev_max)

In [ ]:
samples = []
records = []

for i in range(NUM_SAMPLES):
    crop = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=1000 + i)
    if crop is None:
        print(f"Sample {i + 1}: failed to generate crop")
        continue
    start, end = get_marker_positions(RESOLUTION)
    paths = compute_paths_for_crop(crop, start, end, COST_MODELS_LIST, WEIGHTS)
    samples.append({"index": i + 1, "crop": crop, "start": start, "end": end, "paths": paths})

    for cm in COST_MODELS_LIST:
        for w in WEIGHTS:
            path = paths[cm][w]
            records.append(
                {
                    "sample": i + 1,
                    "cost_model": cm,
                    "weight": w,
                    "path_length": len(path),
                    "found": len(path) > 0,
                }
            )


print(f"\nGenerated {len(samples)}/{NUM_SAMPLES} samples")

In [ ]:
results_df = pd.DataFrame(records)
results_df.head()

In [ ]:
summary_df = (
    results_df.groupby(["cost_model", "weight"])
    .agg(
        mean_path_length=("path_length", "mean"),
        min_path_length=("path_length", "min"),
        max_path_length=("path_length", "max"),
        found_rate=("found", "mean"),
    )
    .reset_index()
)
summary_df

In [ ]:
# Visualizations per sample: rows = cost models, cols = weights
# Note: this will create one figure per sample (20 figures by default).
for sample in samples:
    crop = sample["crop"]
    start = sample["start"]
    end = sample["end"]
    paths = sample["paths"]

    tensor_vis, elev_range = make_tensor_vis(crop, start, end)
    rows = len(COST_MODELS_LIST)
    cols = len(WEIGHTS)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.8 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    for r, cm in enumerate(COST_MODELS_LIST):
        for c, w in enumerate(WEIGHTS):
            ax = axes[r][c]
            path = paths[cm][w]
            img_bgr = tensor_to_image(tensor_vis, path_pixels=path, elev_range=elev_range)
            ax.imshow(img_bgr[:, :, ::-1])
            ax.set_title(f"{cm} | w={w} | len={len(path)}")
            ax.axis("off")

    fig.suptitle(f"Sample {sample['index']}", y=1.02)
    plt.tight_layout()
    plt.show()